In [ ]:
import h5py
import pandas as pd
import numpy as np
import os
from matplotlib import pyplot as plt
import decoding_utils as du
%matplotlib inline 

In [ ]:
stimTableFile = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
stimTable = pd.read_csv(stimTableFile)

In [ ]:
import vbn_utils

In [ ]:
plt.rcParams['font.size'] = 14

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"
unit_table_with_rf_stats = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_rf_stats.csv"
unit_table_opto_metrics = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_opto_metrics.csv"
unit_cluster_labels = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_cluster_labels.csv"
unit_vis_responsive = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_vis_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.4.0/project_metadata/ecephys_sessions.csv"

In [ ]:
units = pd.read_csv(unit_table_file)

In [ ]:
units['cortical_layer'].replace('3-Feb', '2/3', inplace=True)
units['cortical_layer'].replace('6a', '6', inplace=True)
units['cortical_layer'].replace('6b', '6', inplace=True)

units['cortical_layer'].unique()

In [ ]:
stim_table = pd.read_csv(stim_table_file)

In [ ]:
sessions_table = pd.read_csv(sessions_table_file)

In [ ]:
response_units = responses['unitIds'][()]

In [ ]:
g_images = ['omitted'] + list(np.sort(stim_table[(stim_table['stimulus_name'].str.contains('_G_'))&
                    (~stim_table['omitted'])&
                    (~stim_table['image_name'].isin(['im083_r','im111_r']))]['image_name'].unique())) + ['im083_r','im111_r']

h_images = ['omitted'] + list(np.sort(stim_table[(stim_table['stimulus_name'].str.contains('_H_'))&
                    (~stim_table['omitted'])&
                    (~stim_table['image_name'].isin(['im083_r','im111_r']))]['image_name'].unique())) + ['im083_r','im111_r']

In [ ]:
image_dict = {'G': responses.attrs['g_images'], 'H': responses.attrs['h_images']}

In [ ]:
from notebook_utils import (
    calc_pop_sparseness, calc_pop_sparseness_kurtosis,
    get_nonshared_images_from_imageids
)

In [ ]:
from notebook_utils import just_mean, mean_with_base_sub

regions = ['LGd', 'LP', 'VISp', 'VISl', 'VISal', 'VISrl', 'VISpm', 'VISam', 'Hipp', 'SC']
#regions = ['VISp']
window_duration = 50
window_starts = np.arange(0, 350)
window_ends = window_starts + window_duration
methods = [calc_pop_sparseness, calc_pop_sparseness_kurtosis]
baseline_conditions = [just_mean, mean_with_base_sub]
pop_sparseness_by_image = {m: {baseline: {region: {session: {} 
                                for session in sessions_table['ecephys_session_id'].values} 
                                for region in regions} 
                                for baseline in baseline_conditions}
                                for m in methods}
                                
good = du.apply_unit_quality_filter(units)
for ir, session_info in sessions_table.iterrows():
    session_id = session_info['ecephys_session_id']
    session_image_set = session_info['image_set']
    images = image_dict[session_image_set]
    
    print(session_id)

    for region in regions:
        inregion = du.getUnitsInRegion(units, region, cell_type='RS')
        units_to_analyze = units[good&inregion&(units['ecephys_session_id']==session_id)]
        if len(units_to_analyze)<20:
            continue
        response_inds = np.isin(responses['unitIds'][()], units_to_analyze['unit_id'].values)

        for im_ind, im in enumerate(images):
            for method in methods:
                for baseline_cond in baseline_conditions:
                    pop_sparseness_by_image[method][baseline_cond][region][session_id][im] = []
                    im_resp_array = responses['hit'][response_inds, im_ind]
                    baseline = np.mean(im_resp_array[:, :50], axis=1)
                    for slicestart, sliceend in zip(window_starts, window_ends):
                        resp_slice = slice(slicestart, sliceend)
                        resp_vect = baseline_cond(im_resp_array[:, resp_slice], baseline)
                        sparseness = method(resp_vect)

                        pop_sparseness_by_image[method][baseline_cond][region][session_id][im].append(sparseness)

In [ ]:
region = 'VISp'
units_to_analyze = good_units[
                            (good_units['structure_acronym']==region)&
                            (good_units['RS'])&
                            (good_units['firing_rate']>0.1)]
response_inds = np.isin(responses['unitIds'][()], units_to_analyze['unit_id'].values)
im_resp_array = responses['nonchange'][response_inds, im_ind]

In [ ]:
no_abnorm = sessions_table['abnormal_activity'].isnull() & sessions_table['abnormal_histology'].isnull()
# novel_sessions = sessions_table[(sessions_table['experience_level']=='Novel')&(sessions_table['image_set']=='G')]['ecephys_session_id'].values
# familiar_sessions = sessions_table[(sessions_table['experience_level']=='Familiar')&(sessions_table['image_set']=='H')]['ecephys_session_id'].values
novel_sessions = sessions_table[(sessions_table['experience_level']=='Novel')&no_abnorm]['ecephys_session_id'].values
familiar_sessions = sessions_table[(sessions_table['experience_level']=='Familiar')&no_abnorm]['ecephys_session_id'].values

In [ ]:
regions = ['LGd', 'VISp', 'VISl', 'VISrl', 'VISal', 'LP', 'VISpm', 'VISam']
#regions = ['VISp']
for method in methods:
    for baseline_cond in baseline_conditions:
        for region in regions:
            region_condition_sessions = pop_sparseness_by_image[method][baseline_cond][region]
            region_summary_sparseness = {'Novel_unshared':[], 'Novel_shared':[], 'Familiar_unshared':[], 'Familiar_shared':[]}
            for session_id in novel_sessions:
                
                session_ps = pop_sparseness_by_image[method][baseline_cond][region][session_id]
                if len(session_ps.keys())==0:
                    continue
                shared_images = [b'im083_r', b'im111_r']
                non_shared_images = get_nonshared_images_from_imageids(session_ps)

                fam = []
                for im in shared_images:

                    im_sparseness = session_ps[im]
                    fam.append(im_sparseness)
                
                nov = []
                for im in non_shared_images:
                    im_sparseness = session_ps[im]
                    nov.append(im_sparseness)


                region_summary_sparseness['Novel_unshared'].append(np.mean(nov, axis=0))
                region_summary_sparseness['Novel_shared'].append(np.mean(fam, axis=0))

            for session_id in familiar_sessions:
                
                session_ps = pop_sparseness_by_image[method][baseline_cond][region][session_id]
                if len(session_ps.keys())==0:
                    continue
                shared_images = [b'im083_r', b'im111_r']
                non_shared_images = get_nonshared_images_from_imageids(session_ps)

                fam = []
                for im in shared_images:

                    im_sparseness = session_ps[im]
                    fam.append(im_sparseness)
                
                nov = []
                for im in non_shared_images:
                    im_sparseness = session_ps[im]
                    nov.append(im_sparseness)


                region_summary_sparseness['Familiar_unshared'].append(np.mean(nov, axis=0))
                region_summary_sparseness['Familiar_shared'].append(np.mean(fam, axis=0))

            fig, ax = plt.subplots()
            time = np.arange(350)-25
            color_dict = {'Familiar_unshared': 'b', 'Novel_shared': 'purple', 'Novel_unshared': 'r'}
            for key in ['Familiar_unshared', 'Novel_unshared']:
                num_no_nan = np.sum([np.all(~np.isnan(r[50:])) for r in region_summary_sparseness[key]])

                mean = np.nanmean(region_summary_sparseness[key], axis=0)
                sem = np.nanstd(region_summary_sparseness[key], axis=0)/num_no_nan**0.5

                ax.plot(time, mean, color=color_dict[key])
                ax.fill_between(time, mean+sem, mean-sem, color=color_dict[key], alpha=0.3)

            ax.set_title(f'{region} {baseline_cond.__name__} {method.__name__}')
            ax.legend(['f', 'n'])
            ax.set_xlabel('Time from stim start (ms)')
            ax.set_ylabel('Pop Sparseness')

In [ ]:
a = np.load("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vis_responsiveness_over_time/1121406444.npy", allow_pickle=True).item()

In [ ]:
responsiveness_over_time_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vis_responsiveness_over_time"
responsiveness_over_time_files = os.listdir(responsiveness_over_time_dir)
responsiveness_over_time_files = [r for r in responsiveness_over_time_files if 'winsize40' in r]
responsiveness_over_time_sessions = [int(r.split('.npy')[0].split('_')[0]) for r in responsiveness_over_time_files]
responsiveness_over_time_files = [os.path.join(responsiveness_over_time_dir, r) for r in responsiveness_over_time_files]

rot_dict = {}
for rots, rotf in zip(responsiveness_over_time_sessions, responsiveness_over_time_files):
    data = np.load(rotf, allow_pickle=True).item()
    rot_dict[rots] = data


In [ ]:
region_names = ['LGd', 'VISp', 'VISl', 'VISrl', 'VISal', 'LP', 'VISpm', 'VISam', 'Hipp', 'SCMRN', 'VISall']
region_dict = {'LGd': 'LGd', 'VISp':'VISp', 'VISl':'VISl', 'VISrl':'VISrl', 'VISal':'VISal', 'LP':'LP', 'VISpm':'VISpm', 'VISam':'VISam', 
                'Hipp': ['HPF','DG','CA1','CA3'], 'SCMRN':['SCig','SCiw','MRN'], 'VISall': ['VISp','VISl','VISrl','VISal','VISpm','VISam']}

cell_type = 'FS'

for region in region_names:
    region_summary_rot = {'Novel_unshared':[], 'Novel_shared':[], 'Familiar_unshared':[], 'Familiar_shared':[]}
    for session_id in novel_sessions:
        
        session_rot = rot_dict[session_id]
        
        units_to_analyze = vbn_utils.get_unit_ids(good_units, region, cell_types=cell_type, session_id=session_id)
        units_to_analyze = good_units[good_units['unit_id'].isin(units_to_analyze)]
        # units_to_analyze = good_units[(good_units['ecephys_session_id']==session_id)&
        #                             (get_in_region(good_units, region_dict[region]))&
        #                             (good_units[cell_type])]

        if len(units_to_analyze)==0:
            continue
        
        response_inds = np.isin(session_rot['unit_ids'], units_to_analyze['unit_id'].values)

        shared_images = ['im083_r', 'im111_r']
        non_shared_images = [k for k,v in session_rot.items() if 'im' in k and ~np.isin(k, shared_images)]

        fam = []
        for im in shared_images:

            im_sparseness = session_rot[im][response_inds]
            sig = im_sparseness<0.01
            fam.append(np.mean(sig, axis=0))
        
        nov = []
        for im in non_shared_images:
            im_sparseness = session_rot[im][response_inds]
            sig = im_sparseness<0.01
            nov.append(np.mean(sig, axis=0))


        region_summary_rot['Novel_unshared'].append(np.mean(nov, axis=0))
        region_summary_rot['Novel_shared'].append(np.mean(fam, axis=0))

    for session_id in familiar_sessions:
        
        session_rot = rot_dict[session_id]
        
        units_to_analyze = vbn_utils.get_unit_ids(good_units, region, cell_types=cell_type, session_id=session_id)
        units_to_analyze = good_units[good_units['unit_id'].isin(units_to_analyze)]
        # units_to_analyze = good_units[(good_units['ecephys_session_id']==session_id)&
        #                             (get_in_region(good_units, region_dict[region]))&
        #                             (good_units[cell_type])]

        if len(units_to_analyze)==0:
            continue
        
        response_inds = np.isin(session_rot['unit_ids'], units_to_analyze['unit_id'].values)

        shared_images = ['im083_r', 'im111_r']
        non_shared_images = [k for k,v in session_rot.items() if 'im' in k and ~np.isin(k, shared_images)]

        fam = []
        for im in shared_images:

            im_sparseness = session_rot[im][response_inds]
            sig = im_sparseness<0.01
            fam.append(np.mean(sig, axis=0))
        
        nov = []
        for im in non_shared_images:
            im_sparseness = session_rot[im][response_inds]
            sig = im_sparseness<0.01
            nov.append(np.mean(sig, axis=0))


        region_summary_rot['Familiar_unshared'].append(np.mean(nov, axis=0))
        region_summary_rot['Familiar_shared'].append(np.mean(fam, axis=0))

    fig, axes = plt.subplots(1,2)
    ax = axes[0]
    time = np.arange(250)
    color_dict = {'Familiar_unshared': 'b', 'Novel_shared': 'purple', 'Novel_unshared': 'r'}
    for key in ['Familiar_unshared','Novel_unshared']:
        num_no_nan = np.sum([np.all(~np.isnan(r)) for r in region_summary_rot[key]])

        mean = np.nanmean(region_summary_rot[key], axis=0)
        sem = np.nanstd(region_summary_rot[key], axis=0)/num_no_nan**0.5

        ax.plot(time, mean, color=color_dict[key], label=key)
        ax.fill_between(time, mean+sem, mean-sem, color=color_dict[key], alpha=0.3)

        vals = np.stack(region_summary_rot[key])
        for itimepoint, timepoint in enumerate([20, 60]):
            time_mean = np.nanmean(vals[:, timepoint])
            time_sem = np.nanstd(vals[:, timepoint])/num_no_nan**0.5
            axes[1].plot(itimepoint, time_mean, color=color_dict[key], marker='o', markersize=10)
            axes[1].errorbar(itimepoint, time_mean, yerr=time_sem, color=color_dict[key], capsize=5)
    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels(['20-60ms', '60-100ms'])   
    
    

    ax.set_title(region)
    ax.legend()
    ax.set_xlabel('Time from stim start (ms)')
    ax.set_ylabel('Fraction cells responsive')

In [ ]:
aunits = a['unit_ids']
fast = np.min(a['im012_r'][:, 30:60], axis=1)<0.01
notfast = np.min(a['im012_r'][:, 30:60], axis=1)>0.01
fastunits = aunits[fast]
notfastunits = aunits[notfast]
response_inds = np.isin(responses['unitIds'][()], fastunits)
notresponse_inds = np.isin(responses['unitIds'][()], notfastunits)


resps = responses['nonchange'][response_inds, 0]
notresps = responses['nonchange'][notresponse_inds,0]
resps.shape

In [ ]:
plt.plot(np.mean(resps, axis=0))
plt.plot(np.mean(notresps,axis=0))

In [ ]:
plt.plot(np.mean(a['im083_r']<0.01, axis=0))
plt.plot(np.mean(a['omitted']<0.01, axis=0))

In [ ]:
fig, ax = plt.subplots()
for im in images:

    ax.plot(window_starts, pop_sparseness_by_image[im])

### Generated on HPC by 'run_image_decoding.py'

In [ ]:
image_decoding_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding"
decoding_files = os.listdir(image_decoding_dir)
decoding_files = [os.path.join(image_decoding_dir, f) for f in decoding_files if f.endswith('nonchangeRS.npy')]

session_ids = [int(os.path.basename(f).split('_')[0]) for f in decoding_files]

missing_sessions = [s for s in sessions_table['ecephys_session_id'].values if s not in session_ids]
missing_sessions

In [ ]:
decoding_results_files = os.listdir("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding")
decoding_results_files = [os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding", d) for d in decoding_results_files]

In [ ]:
no_anomalies = sessions_table[(sessions_table['abnormal_activity'].isnull())&(sessions_table['abnormal_histology'].isnull())]

In [ ]:
from notebook_utils import (
    get_decoding_results_files, get_mouse_from_session, get_mouse_paired_indices
)

In [ ]:
conditions = ['Familiar', 'Novel']

In [ ]:
image_set = 'nonchangeRS' #Used RS for fig 6
session_decoding_files = get_decoding_results_files(no_anomalies['ecephys_session_id'].values, decoding_results_files, image_set=image_set)
decoding_results_dict = {}
for decoding_file in session_decoding_files:

    d = np.load(decoding_file, allow_pickle=True).item()
    decoding_results_dict[decoding_file] = d

decode_windows = d['decodeWindows']

In [ ]:
gtoh_sessions = no_anomalies[((no_anomalies['image_set']=='G')&(no_anomalies['experience_level']=='Familiar')) | 
                            (((no_anomalies['image_set']=='H')&(no_anomalies['experience_level']=='Novel')))]

htog_sessions = no_anomalies[((no_anomalies['image_set']=='H')&(no_anomalies['experience_level']=='Familiar')) | 
                            (((no_anomalies['image_set']=='G')&(no_anomalies['experience_level']=='Novel')))]

In [ ]:
regions = list(decoding_results_dict[decoding_file].keys())
regions = [r for r in regions if r not in ['decodeWindows', 'hit', 'image_order']]
unit_sub_sample_sizes = list(decoding_results_dict[decoding_file][regions[0]].keys())

conditions = ['Familiar', 'Novel']

region_timecourses = {}
session_ids = {r:{c: {n:[] for n in unit_sub_sample_sizes} for c in conditions} for r in regions}
for region in regions:
    imagewise_recall_per_condition = {c:{} for c in conditions}
    for condition in conditions:

        condition_sessions = no_anomalies[no_anomalies['experience_level']==condition]
        # condition_sessions = htog_sessions[htog_sessions['experience_level']==condition]
        condition_decoding_files = get_decoding_results_files(condition_sessions['ecephys_session_id'].values, decoding_results_files, image_set=image_set)
        
        for n in unit_sub_sample_sizes:
            bas = []
            sess_ids = []
            for cdf in condition_decoding_files:
                d = decoding_results_dict[cdf]
                #d = np.load(cdf, allow_pickle=True).item()
                # ba = np.array(d[region][n]['imagewise_recall'])
                ba = np.array(d[region][n]['imagewise_precision'])
                if ba.size>0:
                    bas.append(ba)
                    sess_ids.append(int(os.path.basename(cdf).split('_')[0]))

            #ba_array = np.array([a for a in bas if a.size>0])
            ba_array = np.array(bas)
            imagewise_recall_per_condition[condition][n] = ba_array
            session_ids[region][condition][n] = sess_ids
        # fig, ax = plt.subplots()
        # ax.plot(np.mean(imagewise_recall_per_condition['Novel'], axis=0))
        # ax.legend(np.arange(8))
        # ax.set_xlim([0,15])
        # fig, ax = plt.subplots()
        # ax.plot(np.mean(imagewise_recall_per_condition['Familiar'], axis=0))
        # ax.legend(np.arange(8))
        # ax.set_xlim([0,15])
    region_timecourses[region] = imagewise_recall_per_condition

In [ ]:
from notebook_utils import (
    fitCurve,
    calc_gompertz,
    invert_gompertz,
    get_sigmoidfit_midpoint,
    find_midpoint_raw
)

In [ ]:
for region in regions:
    fig, axes = plt.subplots(1,len(unit_sub_sample_sizes))
    fig.set_size_inches(18,6)
    imagewise_recall_per_condition = region_timecourses[region]

    for n, ax in zip(unit_sub_sample_sizes, axes):
        # ax.set_title(f'{region} {n}, f: {counts[0]}, n: {counts[1]}')
        colors = ['b', 'r']
        counts = []
        for ic, condition in enumerate(['Familiar', 'Novel']):
            count = imagewise_recall_per_condition[condition][n].shape[0]
            counts.append(count)
            if count>0:
                mean = np.mean(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(0,2))
                std = np.std(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(0,2))
                sem = std/count**0.5
                time = np.arange(0, len(mean)*10, 10)
                ax.plot(time, mean, colors[ic])
                ax.fill_between(time, mean+sem, mean-sem, color=colors[ic], alpha=0.3)
        
        ax.set_title(f'{region} {n}, f: {counts[0]}, n: {counts[1]}')
        ax.set_xlim([0,150])
        ax.set_ylim([0,1])


In [ ]:
#accuracy at end of decision window (100ms) and n=40
hit_rates = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}
for region in regions:
    for n in [40,20]:#unit_sub_sample_sizes[:-1]:
        for condition in conditions:
            timecourse = region_timecourses[region][condition][n]
            count = timecourse.shape[0]
            # counts.append(count)
            if count>0:
                timecourse_means = np.mean(timecourse[:,:,:6], axis=2)
                hrs = timecourse_means[:, 10]
                hit_rates[region][condition][n] = hrs
                

In [ ]:
from notebook_utils import bh_multitest

In [ ]:
regions_to_plot = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'VISall', 'SCMRN']
fig, ax = plt.subplots()
pvals = []
for ir, region in enumerate(regions_to_plot):#regions_to_plot:

    vals = [np.array(hit_rates[region][cond][40]) for cond in ['Familiar', 'Novel']]
    # pval = scipy.stats.mannwhitneyu(*vals, nan_policy='omit')[1]
    pval = scipy.stats.ranksums(*vals, nan_policy='omit')[1]

    pvals.append(pval)
    # print(ir)
    # print(np.nanmean(vals[0]))
    ax.plot(ir, np.nanmean(vals[0]), 'bo')
    ax.plot(ir+0.1, np.nanmean(vals[1]), 'ro')

    ax.errorbar(ir, np.nanmean(vals[0]), np.nanstd(vals[0])/(np.sum(~np.isnan(vals[0]))**0.5), color='b')
    ax.errorbar(ir+0.1, np.nanmean(vals[1]), np.nanstd(vals[1])/(np.sum(~np.isnan(vals[1]))**0.5), color='r')

ax.set_xticks(np.arange(len(regions_to_plot)))
ax.set_xticklabels(regions_to_plot, rotation=90)
ax.set_ylabel('Decoding accuracy')

sig_after_correction = bh_multitest(pvals)[0]
sigx_ind = np.where(sig_after_correction)[0]
if len(sigx_ind)>0:
    ax.text(sigx_ind[0], 1, '*')

vbn_utils.formatFigure(fig, ax)

In [ ]:
colors = ['b', 'r']
plt.rcParams['font.size'] = 20
latencies = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}
latency_sessions = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}

for region in ['LGd', 'VISp', 'VISl', 'VISall']:
    imagewise_recall_per_condition = region_timecourses[region]
    plt.figure()
    fig = plt.gcf()
    fig.set_size_inches([8,6])
    counts=[]
    for n in [40]:
        for ic, condition in enumerate(['Familiar', 'Novel']):
            print(imagewise_recall_per_condition[condition][n].shape)
            count = imagewise_recall_per_condition[condition][n].shape[0]
            counts.append(count)
            if count>0:
                image_means = np.mean(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(2))
                lats = [get_sigmoidfit_midpoint(decode_windows, y)[0] for y in image_means]
                ylats = [get_sigmoidfit_midpoint(decode_windows, y)[1] for y in image_means]
                plt.plot(decode_windows, image_means.T, colors[ic], alpha=0.2)
                plt.plot(lats, ylats, colors[ic]+'o', alpha=0.5, mec='none')
                # sess_ids = [s for s,lat in zip(session_ids[region][condition][n], lats) if ~np.isnan(lat)]
                #lats = [find_midpoint_raw(decode_windows, y)[0] for y in image_means]
                latencies[region][condition][n] = lats
                # latency_sessions[region][condition][n] = sess_ids
                plt.plot(decode_windows, np.mean(image_means, axis=0), colors[ic], lw=2)
                print(f'{region}: {np.median(lats)}')
    ax = plt.gca()
    ax.set_title(f'{region} {n} units, f: {counts[0]}, n: {counts[1]}')
    ax.set_xlim([0, 120])
    ax.set_xticks([20, 100])
    ax.spines['bottom'].set_bounds(20, 100)
    vbn_utils.formatFigure(fig, ax)
    ax.set_xlabel('Time from stim start (ms)')
    ax.set_ylabel('Decoding accuracy')

    # Set the linewidth of all spines
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)  # Set spine linewidth to 2 points

    # Set the linewidth of ticks
    ax.tick_params(width=1.5)

In [ ]:
latencies = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}
latency_sessions = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}

for region in regions:
    imagewise_recall_per_condition = region_timecourses[region]

    for n in [40,20]:#[10,20,40]:
        for ic, condition in enumerate(['Familiar', 'Novel']):
            count = imagewise_recall_per_condition[condition][n].shape[0]
            counts.append(count)
            if count>0:
                image_means = np.mean(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(2))
                lats = [get_sigmoidfit_midpoint(decode_windows, y)[0] for y in image_means]
                # sess_ids = [s for s,lat in zip(session_ids[region][condition][n], lats) if ~np.isnan(lat)]
                #lats = [find_midpoint_raw(decode_windows, y)[0] for y in image_means]
                latencies[region][condition][n] = lats
                # latency_sessions[region][condition][n] = sess_ids

In [ ]:
import scipy.stats
region_diffs = []
for region in ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'VISall']:#regions:

    mouse_match_inds = get_mouse_paired_indices(session_ids, region, 20)
    vals = [np.array(latencies[region][cond][20]) for cond in ['Familiar', 'Novel']]

    mouse_matched = [v[mouse_match_inds[c]] for v,c in zip(vals, ['Familiar', 'Novel'])]

    diffs = mouse_matched[1] - mouse_matched[0]
    region_diffs.append(diffs)
    print(np.median(diffs))
    print(mouse_matched)

fig, ax = plt.subplots()
ax.violinplot(region_diffs)

In [ ]:
regions_to_plot = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'VISall', 'SCMRN']
region_means = []
plt.figure()
pvals = []
for ir, region in enumerate(regions_to_plot):#regions:

    vals = [np.array(latencies[region][cond][20]) for cond in ['Familiar', 'Novel']]
    pvals.append(scipy.stats.mannwhitneyu(*vals, nan_policy='omit')[1])
    # print(ir)
    # print(np.nanmean(vals[0]))
    plt.plot(ir, np.nanmean(vals[0]), 'bo')
    plt.plot(ir+0.1, np.nanmean(vals[1]), 'ro')

    plt.errorbar(ir, np.nanmean(vals[0]), np.nanstd(vals[0])/(np.sum(~np.isnan(vals[0]))**0.5), color='b')
    plt.errorbar(ir+0.1, np.nanmean(vals[1]), np.nanstd(vals[1])/(np.sum(~np.isnan(vals[1]))**0.5), color='r')

ax = plt.gca()
ax.set_xticks(np.arange(len(regions_to_plot)))
ax.set_xticklabels(regions_to_plot, rotation=90)
ax.set_ylabel('Decoding latency (ms)')

sig_after_correction = bh_multitest(pvals)[0]
sigx_ind = np.where(sig_after_correction)[0]
[ax.text(x, ax.get_ylim()[1], '*') for x in sigx_ind]
vbn_utils.formatFigure(fig, ax)

In [ ]:
import ccf_utils
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/ccf_structure_tree_2017.csv")


In [ ]:
unit_sub_sample_sizes = [1, 5, 10, 20, 40, 60, 80]
total_decoding_windows = 30
subsample_balanced_accuracy = {cond: {n:[] for n in unit_sub_sample_sizes} for cond in ['Familiar', 'Novel']}
for condition in conditions:

    condition_sessions = no_anomalies[no_anomalies['experience_level']==condition]
    condition_decoding_files = get_decoding_results_files(condition_sessions['ecephys_session_id'].values, decoding_results_files, image_set='change')
    
    bas = []
    for cdf in condition_decoding_files:
        d = decoding_results_dict[cdf]
        #d = np.load(cdf, allow_pickle=True).item()
        for n in unit_sub_sample_sizes:
            ba = d['VISp'][n]['balanced_accuracy']
            if len(ba)<total_decoding_windows:
                continue
                #ba = np.full(total_decoding_windows, np.nan)
            #ba = np.array([d['VISp'][n]['balanced_accuracy'] for n in unit_sub_sample_sizes])

            subsample_balanced_accuracy[condition][n].append(ba)

In [ ]:
for condition in ['Familiar', 'Novel']:

    for n in unit_sub_sample_sizes:

        ba_array = np.array(subsample_balanced_accuracy[condition][n])
        num_exps = np.array(subsample_balanced_accuracy[condition][n]).shape[0]
        if num_exps>0:

            mean = np.mean(ba_array, axis=0)



In [ ]:
time_point = 5
fig, ax = plt.subplots()
for n in unit_sub_sample_sizes:

    for condition in ['Familiar', 'Novel']:

        ba_array = np.array(subsample_balanced_accuracy[condition][n])
        non_nan_exps = np.unique(np.where(~np.isnan(ba_array))[0])
        if len(non_nan_exps)>0:
            mean = np.mean(ba_array[non_nan_exps], axis=0)
            sem  = np.std(ba_array[non_nan_exps], axis=0)/(len(non_nan_exps))**0.5



In [ ]:
fig,ax = plt.subplots()
for ba_array in [subsample_balanced_accuracy['Familiar'], subsample_balanced_accuracy['Novel']]:
    decoding_curves = []
    for exp in ba_array:
        dcurve = []
        for n in range(len(unit_sub_sample_sizes)):
            vals = exp[n]
            if len(vals)==0:
                val = np.nan
            else:
                val = vals[5]
            dcurve.append(val)
        decoding_curves.append(dcurve)
    decoding_curve_array = np.array(decoding_curves).squeeze().shape

    ax.plot(unit_sub_sample_sizes, np.nanmean(decoding_curves, axis=0))
ax.set_xlabel('unit sample size')
ax.set_ylabel('decoder accuracy at full time')


In [ ]:
import matplotlib.pyplot as plt

# Colors for clusters
colors = [
    '#682F90', '#0252A7', '#0082CA', '#01AAC7', '#01A662',  # Green Family
    '#DC143C', '#FF6347', '#FFA500'  # Purple Family
]
# Plot
fig, ax = plt.subplots(figsize=(10, 2))
for i, color in enumerate(colors):
    ax.bar(i, 1, color=color)

ax.set_xticks(range(len(colors)))
ax.set_xticklabels(colors, rotation=45, ha='right')
ax.set_yticks([])
plt.title("Cluster Colors: Green Family and Purple Family")
plt.show()